# argentina.direcciones — Pruebas interactivas

Recorrido del módulo `argentina.direcciones`.

Funciones simples para normalizar y parsear direcciones argentinas. Solo stdlib — sin geocoding, sin APIs externas, sin pandas.

## 1. Setup

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")

argentina v0.0.19


## 2. normalizar

Lowercase, sin tildes, sin puntuación, espacios colapsados, abreviaturas unificadas.

In [2]:
arg.direcciones.normalizar(" Av. Santa Fe 3253  Piso 2 Depto B ")

'av santa fe 3253 piso 2 depto b'

In [3]:
# Reemplazos típicos
for v in [
    "Avenida Córdoba 1234",
    "Avda. del Libertador 7500",
    "Calle Falsa 123",
    "Pje. Riobamba 456",
    "Maipú 100, Dto 5",
    "San Martín 200 Dpto 3",
    "Departamento principal",
]:
    print(f"{v!r:38} → {arg.direcciones.normalizar(v)!r}")

'Avenida Córdoba 1234'                 → 'av cordoba 1234'
'Avda. del Libertador 7500'            → 'av del libertador 7500'
'Calle Falsa 123'                      → 'falsa 123'
'Pje. Riobamba 456'                    → 'pasaje riobamba 456'
'Maipú 100, Dto 5'                     → 'maipu 100 depto 5'
'San Martín 200 Dpto 3'                → 'san martin 200 depto 3'
'Departamento principal'               → 'depto principal'


In [4]:
# Casos vacíos / None
print(arg.direcciones.normalizar(None))
print(arg.direcciones.normalizar(""))
print(arg.direcciones.normalizar("   "))

None
None
None


## 3. extraer_altura / tiene_altura

Toma el primer grupo numérico de 1 a 5 dígitos.

In [5]:
for v in [
    "Av. Santa Fe 3253",
    "Calle Falsa 123",
    "Av. del Libertador 12345",
    "Sin altura",
    "",
    None,
]:
    print(f"{v!r:30} altura={arg.direcciones.extraer_altura(v)!r:10} tiene={arg.direcciones.tiene_altura(v)}")

'Av. Santa Fe 3253'            altura='3253'     tiene=True
'Calle Falsa 123'              altura='123'      tiene=True
'Av. del Libertador 12345'     altura='12345'    tiene=True
'Sin altura'                   altura=None       tiene=False
''                             altura=None       tiene=False
None                           altura=None       tiene=False


## 4. extraer_calle

Devuelve el texto antes de la altura (sobre la versión normalizada).

In [6]:
for v in [
    "Av. Santa Fe 3253",
    "Avenida Córdoba 1234 Piso 2",
    "Calle Falsa 123 Depto B",
    "Sin altura simplemente",
]:
    print(f"{v!r:38} → {arg.direcciones.extraer_calle(v)!r}")

'Av. Santa Fe 3253'                    → 'av santa fe'
'Avenida Córdoba 1234 Piso 2'          → 'av cordoba'
'Calle Falsa 123 Depto B'              → 'falsa'
'Sin altura simplemente'               → 'sin altura simplemente'


## 5. extraer_piso

Busca `piso N` o `p N` (también `PB`). El número se devuelve en mayúscula como string.

In [7]:
for v in [
    "Av. Santa Fe 3253 Piso 2 Depto B",
    "Maipú 100 piso 12",
    "Corrientes 500 P 3",
    "Lavalle 800 Piso PB",
    "Sin piso explícito 100",
]:
    print(f"{v!r:38} → {arg.direcciones.extraer_piso(v)!r}")

'Av. Santa Fe 3253 Piso 2 Depto B'     → '2'
'Maipú 100 piso 12'                    → '12'
'Corrientes 500 P 3'                   → '3'
'Lavalle 800 Piso PB'                  → 'PB'
'Sin piso explícito 100'               → None


## 6. extraer_departamento

Busca `depto X`, `departamento X`, `unidad X` o `uf X`.

In [8]:
for v in [
    "Av. Santa Fe 3253 Piso 2 Depto B",
    "Maipú 100 dpto 5A",
    "Corrientes 500 unidad 12",
    "Lavalle 800 UF 3",
    "Sin depto 100",
]:
    print(f"{v!r:38} → {arg.direcciones.extraer_departamento(v)!r}")

'Av. Santa Fe 3253 Piso 2 Depto B'     → 'B'
'Maipú 100 dpto 5A'                    → '5A'
'Corrientes 500 unidad 12'             → '12'
'Lavalle 800 UF 3'                     → '3'
'Sin depto 100'                        → '100'


## 7. parsear

Atajo que devuelve un dict con todos los campos.

In [9]:
arg.direcciones.parsear("Av. Santa Fe 3253 Piso 2 Depto B")

{'direccion_normalizada': 'av santa fe 3253 piso 2 depto b',
 'calle': 'av santa fe',
 'altura': '3253',
 'piso': '2',
 'departamento': 'B',
 'tiene_altura': True}

In [10]:
# Sin altura — calle queda con todo, altura=None
arg.direcciones.parsear("Av. Sin Numero")

{'direccion_normalizada': 'av sin numero',
 'calle': 'av sin numero',
 'altura': None,
 'piso': None,
 'departamento': None,
 'tiene_altura': False}

In [11]:
# None
arg.direcciones.parsear(None)

{'direccion_normalizada': None,
 'calle': None,
 'altura': None,
 'piso': None,
 'departamento': None,
 'tiene_altura': False}

## 8. Combinando todo

Lote típico de filas crudas con direcciones de distintos formatos.

In [12]:
registros = [
    " Av. Santa Fe 3253  Piso 2 Depto B ",
    "Avenida Córdoba 1234",
    "Calle Falsa 123, Dpto 5A",
    "Pje. Riobamba 456 P 3",
    "Maipú 100 unidad 12",
    "sin altura",
    None,
    "",
]

for r in registros:
    print(arg.direcciones.parsear(r))

{'direccion_normalizada': 'av santa fe 3253 piso 2 depto b', 'calle': 'av santa fe', 'altura': '3253', 'piso': '2', 'departamento': 'B', 'tiene_altura': True}
{'direccion_normalizada': 'av cordoba 1234', 'calle': 'av cordoba', 'altura': '1234', 'piso': None, 'departamento': None, 'tiene_altura': True}
{'direccion_normalizada': 'falsa 123 depto 5a', 'calle': 'falsa', 'altura': '123', 'piso': None, 'departamento': '5A', 'tiene_altura': True}
{'direccion_normalizada': 'pasaje riobamba 456 p 3', 'calle': 'pasaje riobamba', 'altura': '456', 'piso': '3', 'departamento': None, 'tiene_altura': True}
{'direccion_normalizada': 'maipu 100 unidad 12', 'calle': 'maipu', 'altura': '100', 'piso': None, 'departamento': '12', 'tiene_altura': True}
{'direccion_normalizada': 'sin altura', 'calle': 'sin altura', 'altura': None, 'piso': None, 'departamento': None, 'tiene_altura': False}
{'direccion_normalizada': None, 'calle': None, 'altura': None, 'piso': None, 'departamento': None, 'tiene_altura': False}

## 9. Tests automáticos

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_direcciones.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`re`, `unicodedata`). Sin pandas, sin geocoding, sin APIs externas.
- `extraer_altura` toma el **primer** número de 1–5 dígitos: si la calle empieza con un número (ej. `25 de Mayo 1234`), el `25` queda como altura. Para esos casos hace falta un parser más sofisticado o un dataset de calles oficiales.
- Los reemplazos de abreviaturas son palabra-completa: `"avenida"` se convierte en `"av"`, pero `"avenidas"` (plural) no.
- Para georreferenciación (lat/lon, validación contra padrones de calles) se planea un módulo aparte `arg.geo.direcciones` con dependencias opcionales — fuera de scope por ahora.
- El módulo no intenta detectar entrecalles (`Calle X y Calle Y`); si llega un input así, `extraer_altura` devuelve `None`.